[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/2-data-and-files.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 2. Shop data and files

Tutorial 1 returned one policy section. The model passed a topic. This notebook keeps that tool and adds the shelf.

`get_shop_fact` still maps a topic to a section. The model does not pass a filename. The path check in `tutorial.common.read_file` stays inside that lookup. `query_inventory` reads the shelf created in `tutorial.common.get_db`. The model passes a filter, not a SQL string.

Staff ask which items are low and whether oat milk can ship to a guest. The stock comes from `query_inventory`. The shipping rule comes from `get_shop_fact` with topic `shipping`. A later exchange is staff asking whether a guest can return opened house coffee. Both answers have to come from tools.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.


## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, `setup_colab` in `tutorial/common/colab.py` reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `2-data-and-files.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '2-data-and-files'
LESSON_NUMBER = 2
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
import json

from tutorial.common.facts import get_shop_fact  # registers the tutorial 1 tool
from tutorial.common.harness import assistant_message, check, preview, system_text
from tutorial.common.tools import TOOLS, call_tool
from tutorial.runtime import chat


## Tutorial 2 — the shelf, beside the topic lookup

`query_inventory` reads rows through `tutorial.common.read_db`. The shelf is created in `tutorial.common.get_db`. The SQL stays in `read_db`. The model never writes it. Shipping is still `get_shop_fact` with topic `shipping`, registered by the import above. The model does not choose `policy.md` or `faq.md`.


In [ ]:
import tutorial.common.shelf  # registers query_inventory

from tutorial.common.get_db import reset_db
from tutorial.common.read_db import row_for

reset_db()
oat = row_for("OM-32")
print(oat["name"], "on_hand", oat["on_hand"], "gap", oat["gap"])


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
def run_exchange(user_text):
    """One model call, tool results, second model call. Tutorial 3 wraps this in a loop."""
    messages = [
        {"role": "system", "content": system_text()},
        {"role": "user", "content": user_text},
    ]
    print("QUESTION:", user_text)
    print()
    first = chat(messages, TOOLS)
    if not first["tool_calls"]:
        print("ANSWER:", first["content"])
        print("stop: final (no tool call)")
        return first["content"] or "", []
    messages.append(assistant_message(first))
    log = []
    for call in first["tool_calls"]:
        result = call_tool(call)
        line = call["name"] + " " + json.dumps(call["arguments"], sort_keys=True)
        log.append(line)
        print("[tool]", line)
        print(preview(result))
        print()
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"],
            "content": result,
        })
    second = chat(messages, TOOLS)
    print("ANSWER:", second["content"])
    print("stop: final")
    return second["content"] or "", log


reset_db()
print("=== shelf and policy ===")
answer, log = run_exchange(
    "Which products are at or below their reorder point, and may we ship oat milk to a guest?",
)
check(any(line.startswith("query_inventory") for line in log), "inventory tool ran")
check(any(line.startswith("get_shop_fact") and "shipping" in line.lower() for line in log), "shipping topic was looked up")
check("does not ship" in answer.lower(), "answer uses the shipping rule from the file")
check("HB-12" in answer and "OM-32" in answer, "low-stock skus are named")

print("=== carried forward: one shop fact ===")
fact, fact_log = run_exchange(
    "A guest opened the house coffee—can they return it?",
)
check(any(line.startswith("get_shop_fact") for line in fact_log), "the lesson 1 tool still runs")
check("final sale" in fact.lower(), "the return rule still comes from the tool")
